In [39]:
import pandas as pd
import numpy as np
from scipy.io.wavfile import write

import os
# Creates "Assets" if it doesn't exist; does nothing if it does.
os.makedirs("Assets", exist_ok=True)

In [40]:
def create_openspace_asset(df, cruise_id, vessel_name="Custom Vessel"):
    # 1. Determine time bounds from your data
    # We'll use these for the TimeFrameIntervals
    start_time = pd.to_datetime(df['unixTime'].min(), unit='s').strftime('%Y-%m-%dT%H:%M:%S.00Z')
    end_time = pd.to_datetime(df['unixTime'].max(), unit='s').strftime('%Y-%m-%dT%H:%M:%S.00Z')

    # 2. Define the Lua template
    # Note: We use double curly braces {{ }} for literal Lua tables
    # and single { } for python f-string variables.
    asset_template = f"""
local sun = asset.require("scene/solarsystem/sun/transforms")
local earthTransforms = asset.require("scene/solarsystem/planets/earth/earth")

-- Define the float model resource
local floatModel = asset.resource({{
    Name = "{cruise_id} Model",
    Type = "UrlSynchronization",
    Identifier = "{cruise_id}_model_res",
    -- Url = "https://github.com/CreativeTools/3DBenchy/raw/master/Single-part/3DBenchy.stl",
    Url = "https://raw.githubusercontent.com/kcollins/openspace_rafos/master/oceanographic_argo_profiling_float.glb",
    Version = 1
}})

-- The keyframes for the float's trajectory (generated by openspace_rvdata)
local floatKeyframes = asset.require("./{cruise_id}_keyframes.asset")

-- Define the float's position
local floatPosition = {{
    Identifier = "FloatPosition_{cruise_id}",
    Parent = earthTransforms.Earth.Identifier,
    TimeFrame = {{
        Type = "TimeFrameInterval",
        Start = "{start_time}",
        End = "{end_time}"
    }},
    Transform = {{
        Translation = {{
            Type = "TimelineTranslation",
            Keyframes = floatKeyframes.keyframes
        }}
    }},
    GUI = {{
        Name = "{cruise_id} Position",
        Path = "/Float Tracks/{vessel_name}/{cruise_id}"
    }}
}}

-- Define the float model to be rendered
local floatRenderable = {{
    Identifier = "FloatModel_{cruise_id}",
    Parent = floatPosition.Identifier,
    TimeFrame = {{
        Type = "TimeFrameInterval",
        Start = "{start_time}",
        End = "{end_time}"
    }},
    Transform = {{
        Scale = {{
            Type = "StaticScale",
            Scale = 1000.0
        }}
    }},
    Renderable = {{
        Type = "RenderableModel",
        GeometryFile = floatModel .. "oceanographic_argo_profiling_float.glb",
        LightSources = {{
            sun.LightSource,
            {{
                Identifier = "Camera",
                Type = "CameraLightSource",
                Intensity = 0.5
            }}
        }}
    }},
    GUI = {{
        Name = "{vessel_name} Model",
        Path = "/Float Tracks/{vessel_name}"
    }}
}}

-- Define the trail
local floatTrail = {{
    Identifier = "FloatTrail_{cruise_id}",
    Parent = earthTransforms.Earth.Identifier,
    Renderable = {{
        Type = "RenderableTrailTrajectory",
        Enabled = true,
        Translation = {{
            Type = "TimelineTranslation",
            Keyframes = floatKeyframes.keyframes
        }},
        Color = {{ 1.0, 0.5, 0.0 }},
        StartTime = "{start_time}",
        EndTime = "{end_time}",
        SampleInterval = 60,
        EnableFade = true
    }},
    GUI = {{
        Name = "{cruise_id} Trail",
        Path = "/Float Tracks/{vessel_name}/{cruise_id}",
        Focusable = false
    }}
}}

asset.onInitialize(function()
    openspace.addSceneGraphNode(floatPosition)
    openspace.addSceneGraphNode(floatRenderable)
    openspace.addSceneGraphNode(floatTrail)
end)

asset.onDeinitialize(function()
    openspace.removeSceneGraphNode(floatTrail)
    openspace.removeSceneGraphNode(floatRenderable)
    openspace.removeSceneGraphNode(floatPosition)
end)

asset.export(floatPosition)
asset.export(floatRenderable)
asset.export(floatTrail)
"""

    with open(f"Assets/{cruise_id}.asset", "w") as f:
        f.write(asset_template.strip())

    print(f"Asset created! Ensure {cruise_id}_keyframes.asset exists in the same folder.")



In [41]:
import pandas as pd

def create_openspace_keyframes(df, buoy, cruise):
    """
    Generates an OpenSpace keyframe asset file (.txt) from a pandas DataFrame.
    """
    # Ensure the datetime column is in actual datetime objects
    # Using the 'datetime' column provided in your example
    df['datetime'] = pd.to_datetime(df['datetime'])

    lines = []
    lines.append("local keyframes = {")

    for _, row in df.iterrows():
        # OpenSpace expects ISO format: YYYY-MM-DDTHH:MM:SS
        iso_time = row['datetime'].strftime('%Y-%m-%dT%H:%M:%S')

        entry = f'''  ["{iso_time}"] = {{
    Type = "GlobeTranslation",
    Globe = "Earth",
    Longitude = {row['Longitude']},
    Latitude = {row['Latitude']},
    Altitude = 100,
    UseHeightmap = true
  }},'''
        lines.append(entry)

    lines.append("}\n")
    lines.append('asset.export("keyframes", keyframes)\n')

    # Metadata section
    meta = f'''asset.meta = {{
  Name = "Float Track Position: {buoy}",
  Description = [[This asset provides position information for the float track for the cruise {cruise}: Processed Trackline Navigation Data: One Minute Resolution]],
  Author = "OpenSpace Team",
  URL = "https://www.openspaceproject.com",
  License = "MIT license"
}}'''
    lines.append(meta)

    # Save to file
    filename = f"Assets/{buoy}_keyframes.asset"
    with open(filename, "w") as f:
        f.write("\n".join(lines))

    print(f"Asset file created: {filename}")

# --- Example Usage ---
# df = pd.read_csv("your_data.csv")
# create_openspace_keyframes(df, "Buoy123", "NorthAtlantic2026")

In [42]:
for buoy in ["ISAC2", "ISAC3"]:
    # (NB: I know they're floats, not buoys -- but "float" is a reserved word!)
    filename = buoy+".csv"
    df = pd.read_csv(filename)
    # df = df.dropna() # drop any NaN rows
    # Convert to datetime
    # df['datetime'] = pd.to_datetime(df['Date (yyyy-MMM-dd hh:mm)'], format='mixed', utc=True)
    # df['unixTime'] = df['datetime'].astype('int64') // 10**6
    # To verify, print the first value:
    print(f"Verified Unix Time: {df['unixTime'].iloc[0]}")
    df = df.rename(columns={"Datetime": "datetime"})
    df['datetime'] = pd.to_datetime(df['datetime'])
    df= df.resample('1min', on='datetime').mean(numeric_only=True)
    df = df.reset_index()
    # Create OpenSpace asset files:
    print(f"Creating asset file for {buoy}:............................")
    create_openspace_asset(df, buoy, "PACSUN2603")
    print(f"Creating keyframe asset for {buoy}:............................")
    create_openspace_keyframes(df, buoy, "PACSUN2603")
    # Generate .wav file:
    # print(f"Sonifying data from {buoy}:............................")


Verified Unix Time: 1774047600
Creating asset file for ISAC2:............................
Asset created! Ensure ISAC2_keyframes.asset exists in the same folder.
Creating keyframe asset for ISAC2:............................
Asset file created: Assets/ISAC2_keyframes.asset
Verified Unix Time: 1774141980
Creating asset file for ISAC3:............................
Asset created! Ensure ISAC3_keyframes.asset exists in the same folder.
Creating keyframe asset for ISAC3:............................
Asset file created: Assets/ISAC3_keyframes.asset


In [43]:
df.head()

,datetime,unixTime,background_113,ambient_113,backscatter_113,pressure_113,waterTemp_113,battery_113,backscatter_filtered_113,ambient_filtered_113,...,CommId,AgeInSeconds,BatteryVoltage,GpsQuality,Latitude,Longitude,SubmergedBoolean,Temperature0cm,Unnamed: 11,station
0,2026-03-22 14:13:00+13:00,1.774142e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.010000e+14,3618.0,10.5,3.0,-21.099825,-175.713927,0.0,NaN,NaN,121.0
1,2026-03-22 14:14:00+13:00,1.774142e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.010000e+14,3618.0,10.5,3.0,-21.099825,-175.713927,0.0,NaN,NaN,121.0
2,2026-03-22 14:15:00+13:00,1.774142e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.010000e+14,4509.0,10.5,3.0,-21.081147,-175.717979,0.0,NaN,NaN,120.0
3,2026-03-22 14:16:00+13:00,1.774142e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.010000e+14,4509.0,10.5,3.0,-21.081147,-175.717979,0.0,NaN,NaN,120.0
4,2026-03-22 14:17:00+13:00,1.774142e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.010000e+14,4509.0,10.5,3.0,-21.081147,-175.717979,0.0,NaN,NaN,120.0
